# Task 2
Create a 3D-graph with "center of molecule" as the node and distance to its neighbouring molecules as the edges

## Get data from npt-HK4.gro file

In [12]:
# imports
import pandas as pd
import numpy as np
"""
Reads the npt-HK4.gro and returns the following:

data(pd.Dataframe): A dataframe that contains Residue ID(res_id) and Name(res_name), Atom Name (atom_name) and ID(atom_id), the atom coordinates(x, y, z) and velocity components(Vx, Vy, Vz)
title(str): The title of the .gro file
num_atoms(str): The number of atoms
box_dimensions(list): The simulation box dimensions in a list of [x,y,z] 

"""
# setting column specifications
colspecs = [
    (0, 5),
    (5, 8),
    (8, 15),
    (15, 20),
    (20, 28),
    (28, 36),
    (36, 44),
    (44, 52),
    (52, 60),
    (60, 68),
]

# setting names of columns
names = ["res_id", "res_name", "atom_name", "atom_id", "x", "y", "z", "Vx", "Vy", "Vz"]

# reading data
data = pd.read_fwf(
    "../../data/npt-HK4.gro",
    colspecs=colspecs,
    names=names,
    skiprows=2,
    skipfooter=1,
)

# Problem: after atom id 99999, it reverts back to 0. The following solves this:
for i in data.index:
    if i >= 99999:
        data.at[i, "atom_id"] += 100000

# reading Title, number of atoms and box_dimensions
with open("../../data/npt-HK4.gro", "rb") as f:
    title = f.readline().decode().strip()  # First line of .gro file
    num_atoms = f.readline().decode().strip()  # Second line of .gro file

    # Last line of file
    f.seek(-2, 2)
    while f.read(1) != b"\n":
        f.seek(-2, 1)
    box_dimensions = f.readline().decode().strip().split()
    box_dimensions = list(map(float, box_dimensions))


print(data.tail())
print(f"Title: {title}")
print(f"Number of atoms: {num_atoms}")
print(f"Box dimensions: {box_dimensions}")

        res_id res_name atom_name  atom_id      x      y      z      Vx  \
126079    1501      HK4       H23   126080  0.953  5.984  1.494 -1.2440   
126080    1501      HK4       C44   126081  0.736  6.009  1.525  0.1513   
126081    1501      HK4       H24   126082  0.707  6.041  1.425  0.3100   
126082    1501      HK4       C45   126083  0.632  5.988  1.615 -0.0110   
126083    1501      HK4       H25   126084  0.542  6.051  1.607 -0.7669   

            Vy      Vz  
126079  0.1068 -0.7226  
126080  0.5200  0.5632  
126081  2.4172  1.1123  
126082 -0.1637  0.2235  
126083 -0.9514  2.2265  
Title: mol only system in water
Number of atoms: 126084
Box dimensions: [11.24798, 11.24798, 11.24798]


## Midpoint finding algorithm to find the "center of the molecule", defined as the midpoint of both oxygen atoms in the molecule

In [13]:
"""
NEED CHANGING TO MINIMUM IMAGE CONVENTION METHOD
"""

# Get all oxygen atoms location
O1 = data.loc[data["atom_name"] == "O1", ["x", "y", "z"]].reset_index(drop=True)
O2 = data.loc[data["atom_name"] == "O2", ["x", "y", "z"]].reset_index(drop=True)

#Checking for periodic boundary conditions
for i in O1.index:
    diff = np.ndarray.tolist(O1.subtract(O2).values[0])
    for x in diff:
        if x > 1:
            xyz = diff.index(x)
            match xyz:
                case 0:
                    O1.loc[i,'x'] = O1.at[i,'x'] - box_dimensions[xyz]
                case 1:
                    O1.loc[i,'y'] = O1.at[i,'y'] - box_dimensions[xyz]
                case 2:
                    O1.loc[i,'z'] = O1.at[i,'z'] - box_dimensions[xyz]
        elif x < -1:
            xyz = diff.index(x)
            match xyz:
                case 0:
                    O2.loc[i,'x'] = O2.at[i,'x'] - box_dimensions[xyz]
                case 1:
                    O2.loc[i,'y'] = O2.at[i,'y'] - box_dimensions[xyz]
                case 2:
                    O2.loc[i,'z'] = O2.at[i,'z'] - box_dimensions[xyz]

#Getting midpoint and put into a dataframe "centre"
centre = O1.add(O2).abs().divide(2)
centre.insert(0,"res_id", data["res_id"].unique().astype(int))

centre
# Nitrogen locations(FOR FUTURE USE)
# N1 = data.loc[data['atom_name'] == "N1",['x','y','z']].reset_index(drop=True)
# N2 = data.loc[data['atom_name'] == "N2",['x','y','z']].reset_index(drop=True)

,res_id,x,y,z
0,1,1.5585,0.4090,0.07849
1,2,0.4415,0.6570,10.78750
2,3,0.7645,10.7200,0.89800
3,4,5.7815,0.9775,0.13500
4,5,5.5440,5.7345,5.77600
...,...,...,...,...
1496,1497,8.5520,6.7850,8.62900
1497,1498,8.6045,0.9360,7.08450
1498,1499,5.2595,0.5265,5.62250
1499,1500,5.9295,10.4510,3.18150


## Made subboxes. To define which molecules are "near" each other.
Molecules are near each other if:
- They exist within the same subbox
- They are in adjacent subboxes

Each subbox should contain 1 molecule on average.

In [14]:
## First, find dimension of one molecule to approximate number of subboxes needed
molecule_size = pd.DataFrame(columns=["res_id", "molecule_size"])
for n in range(1, 1502):
    one_molecule = data.loc[data["res_id"] == n, ["x", "y", "z"]]
    molecule_dimensions = (one_molecule.max() - one_molecule.min()).to_list()
    molecule_size.loc[n - 1] = [n, np.prod(molecule_dimensions)]

### My first attempt, but this is number of subboxes per axis, could be useful
# number_of_subboxes = [round(x / y) for x,y in zip(box_dimensions,molecule_dimensions)]
# number_of_subboxes

number_of_subboxes = round(np.prod(box_dimensions) / np.prod(molecule_dimensions))
## Output: number_of_subboxes = 142

## Subbox dimension = box_dimension/mol_dimension + some arbitrary constant to make it slightly bigger to lower chance of edge case where no molecule exist in a subbox
## Use the closest cube root(ie. 125)to provide a better approximation as well as to make calculations simpler

####LOGIC ERROR: WE USE 1501 SUBBOXES, BUT SINCE 1501 IS NOT A PERFECT CUBE, USE 1331 SINCE NEAREST PERFECT CUBE

In [15]:
"""
We self define the number of subboxes needed as 1331 as the total number of molecules is 1501 and we take the closest cube root to it.

Returns:
number_of_subboxes(int): number of subboxes needed
subboxes_dimensions(list): A list of dimensions of a subbox
"""


number_of_subboxes = 1331  #Define number of subboxes needed
subboxes_dimensions = [x / number_of_subboxes ** (1 / 3) for x in box_dimensions] #get subbox dimensions and put into a list

In [16]:
"""
With the dimensions, we create a dataframe of each subbox with the nodes(defined as center of molecule) in the subbox

Returns:
subbox(pd.Dataframe): A dataframe that contains the Index of the Subbox as a Tuple(index), Residue ID(res_ID), the Center of Molecule coordinates(x, y, z)

"""



# Make a list of all possible indices of subboxes, additionally with an outer layer
matrix = [] #Empty list to append into later
n = round(number_of_subboxes ** (1 / 3)) # Maximum indices for the box, excluding outer layer
# Nested for loop to create combinations of 3 from a set of numbers
for i in range(1, n + 1):
    for j in range(1, n + 1):
        for k in range(1, n + 1):
            matrix.append([i, j, k])

# Find molecules within a subbox and insert it as a new row into a dataframe
subbox = pd.DataFrame(columns=["index", "res_id", "x", "y", "z"])
iterate = 1
for p, q, r in matrix:
    molecule_in_subbox = centre.loc[
        (centre["x"] < subboxes_dimensions[0] * p)
        & (centre["y"] < subboxes_dimensions[1] * q)
        & (centre["z"] < subboxes_dimensions[2] * r)
        & (centre["x"] >= subboxes_dimensions[0] * (p - 1))
        & (centre["y"] >= subboxes_dimensions[1] * (q - 1))
        & (centre["z"] >= subboxes_dimensions[2] * (r - 1))
    ]

    subbox.loc[iterate] = {
        "index": (p, q, r),
        "res_id": molecule_in_subbox["res_id"].values,
        "x": molecule_in_subbox["x"].values,
        "y": molecule_in_subbox["y"].values,
        "z": molecule_in_subbox["z"].values,
    }
    iterate += 1

subbox

,index,res_id,x,y,z
1,"(1, 1, 1)",[],[],[],[]
2,"(1, 1, 2)",[],[],[],[]
3,"(1, 1, 3)",[979],[0.764],[0.7645],[2.083]
4,"(1, 1, 4)","[977, 1171]","[0.18, 0.5825]","[0.246, 0.5185]","[3.944, 3.33]"
5,"(1, 1, 5)",[343],[0.199],[0.5265],[5.0825]
...,...,...,...,...,...
1327,"(11, 11, 7)",[],[],[],[]
1328,"(11, 11, 8)",[1350],[10.836500000000001],[10.552],[8.15]
1329,"(11, 11, 9)",[1442],[10.852],[10.82],[8.9085]
1330,"(11, 11, 10)",[1013],[10.664],[10.6395],[9.4485]


In [82]:
"""
Make a new multiindexed dataframe that contains the subbox with their respective adjacent subboxes and the Molecules within each adjacent subbox

Returns:
edges_df(pd.Dataframe): A Multiindex Dataframe that contains the subboxes(select_index), its respective adjacent subboxes(adjacent_index), and the nodes within the subboxes(res_id) 

"""
# Create empty lists to append to
edges_res_id = []
edges_index = []

# Select a box
for n in subbox.index:
    selected_box = subbox.at[n,"index"] 
    selected_box_str = ",".join(map(str,selected_box))
# Get indices of adjacent boxes to the selected box
    adjacent_boxes = []
    for (p,q,r) in [list(selected_box)]:
        for i in [-1,0,1]:
            for j in [-1,0,1]:
                for k in [-1,0,1]:
                    x = p + i
                    y = q + j
                    z = r + k
                    match x:
                        case 0:
                            x = 11
                        case 12:
                            x = 1
                        case _:
                            x = x
                    match y:
                        case 0:
                            y = 11
                        case 12:
                            y = 1
                        case _:
                            y = y
                    match z:
                        case 0:
                            z = 11
                        case 12:
                            z = 1
                        case _:
                            z = z
                    adjacent_boxes.append((x,y,z))
        # adjacent_boxes.remove((p,q,r)) # Uncomment this line to exclude itself from adjacent_boxes
# For loop iterating over each adjacent box to find the nodes inside them, then append into list
    for (p,q,r) in adjacent_boxes:
        adjacent_box_str = ",".join(map(str,(p,q,r)))
        adjacent_nodes = subbox.loc[subbox["index"] == (p, q, r), "res_id"]
        if adjacent_nodes.empty:
                    continue
        edges_res_id.append(adjacent_nodes.values[0]) 
        edges_index.append((selected_box_str,adjacent_box_str))

edges_df = pd.DataFrame(data = {'res_id': edges_res_id}, index = pd.MultiIndex.from_tuples(edges_index, names=['select_index', 'adjacent_index']))
edges_df.head(28)

res_id
select_index adjacent_index              
1,1,1        11,11,11              [1181]
             11,11,1                   []
             11,11,2                [147]
             11,1,11           [127, 780]
             11,1,1                    []
             11,1,2                [1366]
             11,2,11           [274, 660]
             11,2,1                 [226]
             11,2,2           [897, 1202]
             1,11,11                [850]
             1,11,1                   [3]
             1,11,2                [1380]
             1,1,11                   [2]
             1,1,1                     []
             1,1,2                     []
             1,2,11           [984, 1343]
             1,2,1           [1206, 1330]
             1,2,2                     []
             2,11,11                [943]
             2,11,1                [1088]
             2,11,2                    []
             2,1,11            [164, 244]
             2,1,1               [1, 847]
             2,1,2                     []
             2,2,11                [1055]
             2,2,1                  [359]
             2,2,2                  [247]
1,1,2        11,11,1                   []

# Trying out Gabriel Graph

In [7]:
"""
Gabriel Graph

Has significantly less edges than the edges algorithm we did. This is because they find different things, for gabriel graphs give the nearest neighbours 
by defining that no other vertex can exist between a region between two vertices. We define it as all vertices within a region of 3x3 subbox are considered
near to each other.

Gabriel Graph however does not give us which points are closest to a given edge, making it so that any blocking algorithm we make would have to iterate over all nodes
within the system. The subbox method on the other hand tells us which nodes are in the vicinity of a given edge, therefore minimizing computation time.

"""

import itertools
import networkx as nx
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay

# Trying Gabriel Graph
def DelaunayGraph(points):
    tri = Delaunay(points)
    G = nx.Graph()

    indptr = tri.vertex_neighbor_vertices[0]
    indices = tri.vertex_neighbor_vertices[1]
    for i in range(len(points)):
        for j in indices[indptr[i]:indptr[i+1]]:
            if i < j:
                G.add_edge(i,j)

    return G

points = centre[['x','y','z']].values
gabrielGraph = DelaunayGraph(points)

for e in gabrielGraph.edges():
    for other in gabrielGraph.nodes():
        if other not in e:
            p1 = points[e[0]]
            p2 = points[e[1]]
            p3 = points[other]
            center = 0.5 * (p1 + p2)
            radius = 0.5 * np.linalg.norm(p2-p1)
            if (np.linalg.norm(p3-center) <= radius):
                gabrielGraph.remove_edge(e[0],e[1])
                break



In [ ]:
# #Making edges
# edges = []
# new_edges = []

# ## Step 1: Connect edges nodes within subbox
# def generate_combinations(items, x):
#     result = []
    
#     def backtrack(start, current):
#         if len(current) == x:
#             result.append(tuple(current))
#             return
#         for i in range(start, len(items)):
#             current.append(items[i])
#             backtrack(i + 1, current)
#             current.pop()
    
#     backtrack(0, [])
#     return result

# for n in subbox.index:
#     selected_box = [list(subbox.at[n,"index"])]
#     nodes = subbox.at[n,"res_id"]
#     if len(nodes) > 1:
#         combin = generate_combinations(nodes,2)
#         for (x,y) in combin:
#             # if ((x,y) not in edges or (y,x) not in edges):
#             edges.append((x,y))
# ## Step 2: Find adjacent boxes
#     adjacent_boxes = []
#     for (p,q,r) in selected_box:
#         for i in [-1,0,1]:
#             for j in [-1,0,1]:
#                 for k in [-1,0,1]:
#                     adjacent_boxes.append((p+i,q+j,r+k))
#         adjacent_boxes.remove((p,q,r))
# ## Step 3: Connect nodes in selected subbox to nodes in adjacent subbox 
#         for (p,q,r) in adjacent_boxes:
#             try: 
#                 adjacent_nodes = subbox.loc[subbox["index"] == (p, q, r), "res_id"]
#                 if adjacent_nodes.empty:
#                     continue
#                 adjacent_nodes = adjacent_nodes.values[0]
#             except:
#                 print(f"fail at {(p,q,r)}")

#             for x in adjacent_nodes:
#                 for j in nodes:
#                     if ([j,x] not in new_edges or [x,j] not in new_edges):
#                         new_edges.append((j, x)) 


# edges = [tuple(map(int,group)) for group in edges]
# new_edges = [tuple(map(int,group)) for group in new_edges]

# total_edges = edges + new_edges
# len(total_edges)

38199

In [9]:
# #Uncomment the following to visualize the edges
# import networkx as nx
# import matplotlib.pyplot as plt

# # Create graph from all edges
# G = nx.Graph(edges)
# H = nx.Graph(new_edges)
# H.add_edges_from(edges)

# # Plot the graphs
# plt.figure(figsize=(12, 8))
# plt.subplot(121)   

# plt.title("Edges within Subbox")
# plt.subplot(122)
# nx.draw(H, with_labels=False, node_size=1, edge_color='red', width=0.1)
# plt.title("Edges between Subboxes")

In [ ]:
# ## Step 4: assign weightage to each edge
# total_edges = edges + new_edges # Can change this later to only have "edges", I made new_edges to show the step by step progression of the number of edges

# #blocking code

# for (p,q) in total_edges: 
#     molecule_1 = centre.loc[centre["res_id"] == p, ['x','y','z']].values[0]
#     molecule_2 = centre.loc[centre["res_id"] == q, ['x','y','z']].values[0]
#     distance = float(np.sum(np.square(np.add(molecule_1,molecule_2))) ** (1/2))
#     total_edges[total_edges.index((p,q))] = (p,q, distance)

# len(total_edges)

38199

In [11]:
# import networkx as nx
# Weighted_graph = nx.Graph()
# Weighted_graph.add_weighted_edges_from(total_edges)
# nx.draw(Weighted_graph, with_labels=False, node_size=1, edge_color='red', width=0.1)
# plt.title("Graph with weighted edges")